# Loose coupling of FreeGSNKE with TORAX: resolving the vessel dynamics

Example 12a couples TORAX to *static* FreeGSNKE equilibria: over each coupling interval the coil currents are prescribed and the vessel eddy currents are assumed to have decayed (the quasi-static approximation). This notebook uses `torax_coupling.EvolutiveEquilibriumSolver` instead, which integrates the **coil and passive-structure circuit equations on the vessel timescale** inside every transport coupling interval:

- The transport code owns the slow variables: kinetic profiles, current diffusion and the total plasma current $I_p$. Over a coupling interval $[t, t+\Delta t]$ it provides $p'(\psi)$, $FF'(\psi)$ and $I_p$ at both ends (as IMAS `equilibrium` IDSs), which FreeGSNKE interpolates linearly in time.
- FreeGSNKE owns the fast variables: the currents in the active coils and in the passive structures, driven by the applied coil voltages, are advanced with sub-millisecond implicit Euler steps of the circuit equations (`freegsnke.metal_evolution.MetalCurrentsEvolution`). Passive-structure normal modes decaying faster than the sub-step are dropped, as in FreeGSNKE's own evolutive solver. At each sub-step the equilibrium is the static free-boundary solution for the instantaneous currents and profiles, so the plasma's vertical motion and the eddy currents are resolved.
- Each implicit sub-step is solved with FreeGSNKE's Newton-Krylov solver on the small vector of metal mode currents (the plain fixed-point iteration between circuit equations and equilibrium converges poorly because, on timescales shorter than the vessel L/R times, the passive structures respond to plasma motion with order-one image currents).

Note that FreeGSNKE's `nonlinear_solve` evolutive solver is **not** used: it evolves the plasma current with a lumped plasma circuit equation and a parametric profile family, which would compete with TORAX's current diffusion.

**Vertical stability.** The MAST-U-like plasma used here is vertically unstable (growth time of a few milliseconds), so left alone it drifts to the wall within a coupling interval. The vertical position is therefore controlled: the circuit equation of the P6 coil is replaced by the constraint that the magnetic axis stays at its initial height (`vertical_control=(coil_index, z_target)`, an ideal fast controller; the voltage it would need is reported). A voltage feedback controller (`metal_evolution.VerticalPositionController`) is also available, but its gains have to be tuned against the *dynamic* response of the plasma, which for a coil outside the passive structures can initially have the opposite sign to the static one.

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt

import torax
from freegsnke import build_machine, equilibrium_update, GSstaticsolver, torax_coupling
from freegsnke.jtor_update import ConstrainPaxisIp

## Initial FreeGSNKE equilibrium (as in example 12a)

In [ ]:
tokamak = build_machine.tokamak(
    active_coils_path="../machine_configs/MAST-U/MAST-U_like_active_coils.pickle",
    passive_coils_path="../machine_configs/MAST-U/MAST-U_like_passive_coils.pickle",
    limiter_path="../machine_configs/MAST-U/MAST-U_like_limiter.pickle",
    wall_path="../machine_configs/MAST-U/MAST-U_like_wall.pickle",
)
eq = equilibrium_update.Equilibrium(tokamak=tokamak, Rmin=0.1, Rmax=2.0, Zmin=-2.2, Zmax=2.2, nx=65, ny=129)
profiles = ConstrainPaxisIp(eq=eq, paxis=8e3, Ip=6e5, fvac=0.5, alpha_m=1.8, alpha_n=1.2)
with open("data/simple_diverted_currents_PaxisIp.pk", "rb") as f:
    currents = pickle.load(f)
for label, current in currents.items():
    eq.tokamak.set_coil_current(coil_label=label, current_value=current)
solver = GSstaticsolver.NKGSsolver(eq)
solver.solve(eq=eq, profiles=profiles, constrain=None, target_relative_tolerance=1e-8)
print("active coils:", tokamak.coils_list[: tokamak.n_active_coils])
print("passive structures:", tokamak.n_coils - tokamak.n_active_coils)

## TORAX configuration (as in example 12a)

In [ ]:
def parabolic(axis_value, edge_value, n_points=21):
    rho = np.linspace(0.0, 1.0, n_points)
    return {float(r): float(edge_value + (axis_value - edge_value) * (1 - r**2)) for r in rho}

CONFIG = {
    "profile_conditions": {
        "Ip": 6e5,
        "T_i": {0.0: parabolic(0.6, 0.1)}, "T_e": {0.0: parabolic(0.6, 0.1)},
        "T_i_right_bc": 0.1, "T_e_right_bc": 0.1,
        "n_e": {0.0: parabolic(3.9e19, 1.8e19)}, "nbar": 3e19, "n_e_nbar_is_fGW": False,
        "n_e_right_bc": 1e19, "n_e_right_bc_is_fGW": False,
        "initial_psi_mode": "geometry",
    },
    "plasma_composition": {},
    "numerics": {"t_initial": 0.0, "t_final": 0.02, "fixed_dt": 0.005, "adaptive_dt": False, "evolve_current": True, "evolve_density": False},
    "geometry": {"geometry_type": "circular", "n_rho": 25, "R_major": 0.87, "a_minor": 0.53, "B_0": 0.58},
    "neoclassical": {"bootstrap_current": {}},
    "sources": {"generic_heat": {"P_total": 3e5}, "ei_exchange": {}, "ohmic": {}},
    "transport": {"model_name": "combined", "transport_models": [{"model_name": "constant"}]},
    "solver": {"solver_type": "linear"},
    "pedestal": {},
    "time_step_calculator": {"calculator_type": "fixed"},
}
torax_config = torax.ToraxConfig.from_dict(CONFIG)

## The evolutive equilibrium provider

The coil voltages default to the steady-state values $U = R I$ of the initial currents, so that without the plasma the coil currents would stay constant; any callable `active_voltages(time, evolution)` can be used instead. P6 (index 11) is the vertical control coil.

In [ ]:
P6 = tokamak.coils_list.index("P6")
equilibrium_solver = torax_coupling.EvolutiveEquilibriumSolver(
    eq,
    profiles,
    solver=solver,
    vessel_timestep=5e-4,                        # sub-step of the circuit equations [s]
    vertical_control=(P6, float(eq.opt[0][1])),  # hold the magnetic axis at its initial height
    verbose=False,
)
print("metal modes retained:", equilibrium_solver.evolution.n_modes, "of", tokamak.n_coils)

## Running the coupled simulation

Each coupling interval of 10 ms is iterated as in example 12a; every iteration re-integrates the vessel dynamics over the interval from the last committed state (20 sub-steps, each a Newton-Krylov solve with a few static equilibrium solves), so this takes considerably longer than the static coupling.

In [ ]:
result = torax_coupling.run_loose_coupling(
    torax_config,
    equilibrium_solver,
    coupling_dt=0.01,
    max_iterations=8,
    tolerance=2e-3,
    relaxation=0.5,
)
print("TORAX error state:", result.sim_error)
print("coupling times:", result.times)
print("iterations per coupling time:", result.iterations, "converged:", result.converged)

## Results

`equilibrium_solver.substep_history` holds the committed vessel-timescale history: all metal currents, the plasma current, the magnetic axis position and (for the controlled coil) the implied voltage at every sub-step.

In [ ]:
history = equilibrium_solver.substep_history
t_ms = np.array([h["time"] for h in history]) * 1e3
n_active = tokamak.n_active_coils

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
axes[0, 0].plot(t_ms, [h["Ip"] / 1e3 for h in history])
axes[0, 0].set_ylabel("$I_p$ [kA]")
axes[0, 1].plot(t_ms, [h["Z_axis"] * 1e3 for h in history], label="$Z_{axis}$")
axes[0, 1].plot(t_ms, [(h["R_axis"] - history[0]["R_axis"]) * 1e3 for h in history], label="$R_{axis} - R_{axis}(0)$")
axes[0, 1].set_ylabel("[mm]"); axes[0, 1].legend(fontsize=8)
axes[1, 0].plot(t_ms, [np.abs(h["currents"][n_active:]).sum() / 1e3 for h in history])
axes[1, 0].set_ylabel("total |passive current| [kA]")
axes[1, 1].plot(t_ms, [h["currents"][P6] for h in history], label="$I_{P6}$ [A]")
axes[1, 1].plot(t_ms, [h["implied_voltage"] for h in history], label="$U_{P6}$ [V] (implied)")
axes[1, 1].legend(fontsize=8)
for ax in axes.ravel():
    ax.set_xlabel("t [ms]")
plt.tight_layout()

In [ ]:
profiles_out = result.torax_output["profiles"]
rho = profiles_out["rho_norm"].values
rho_face = profiles_out["rho_face_norm"].values   # q is defined on the face grid
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for t in result.times:
    # the output holds every TORAX step: select the one at the coupling time
    i = int(np.argmin(np.abs(profiles_out["time"].values - t)))
    axes[0].plot(rho, profiles_out["T_e"].values[i], label=f"t = {t:.2f} s")
    axes[1].plot(rho_face, profiles_out["q"].values[i])
axes[0].set_ylabel("$T_e$ [keV]"); axes[0].legend(fontsize=8); axes[1].set_ylabel("q")
for ax in axes:
    ax.set_xlabel(r"$\rho_N$")
plt.tight_layout()